# 🫀 퀘스트 46 · Q4-K — **P-P / PR / 구간비**: 이소성 초점을 시간으로 잡는다

| | **MedKOS / `notebooks/quest46_q4k_pp_interval.ipynb`** |
|---|---|
| 퀘스트 | `ailab-2026-0046` — 층① 표현, **RR 축을 떠난다** |
| 부모 런 | `quest46_q4j_error_anatomy`(`20260805T0914`) |
| 성격 | 사용자 임상 가설(P-P·PR·구간비)을 **수치화**하고, 내 지표 선택 오류를 고친다 |
| 예상 소요 | **30–40분**(CPU) + DL 절 **10–20분**(GPU 있을 때만) |

## ★★★ Q4-J 는 성공했는데 내가 잘못된 자로 쟀다

```
지표             base     morph     Δ        상대
매크로 AUROC     0.9420   0.9529   +0.0109   +1.2%   ← 내가 주 관문으로 건 것
매크로 PR-AUC    0.5796   0.7236   +0.1440  +24.8%
민감도@300       0.7459   0.8211   +0.0752  +10.1%
PPV@300         0.3620   0.4280   +0.0660  +18.2%
달성률@300       0.8074   0.9018   +0.0944  +11.7%
```

주 관문 `morph − mshuf` 는 **AUROC 로 +0.0142 [+0.0012, +0.0273] · 문턱 +0.0248 → ⚠️ 미결**
이었다. 그런데 **같은 개입이 달성률을 +0.0944 올렸다.**

**왜 갈렸나 — 개입이 작용하는 자리가 다르다.** 형태는 **상위 300개에서 V 를 걷어내는**
일을 한다(V 위양성 **4380 → 1444 · 67% 감소**, N 위양성은 6338 → 8166 으로 **빈자리를
채웠다**). 그건 **순위의 맨 위**에서 벌어지는 일이고, AUROC 는 전체 순위쌍에 지배되므로
거의 안 움직인다. **Q4-G 의 ρ(AUROC, 달성률)=+0.8869 는 「레코드 간 상관」이지
「개입이 둘을 같이 움직인다」가 아니다 — 내가 둘을 혼동했다.**

⇒ **사전등록 변경(데이터 보기 전)**: 주 지표를 **달성률@300**(1차) + **매크로 AUROC**(2차)
**공동**으로 바꾼다.

## ★★★ N1 오류 해부가 알려준 것

```
위양성 10,718개 — N 6,338(59.1%) · V 4,380(40.9%)
위음성  5,896개 — 불규칙(AF 대리) 상위 5,306(90.0%) vs 하위 590
       런/이단맥 39.7% · **놓친 S 의 상대 RR 중앙 0.868** · 점수 백분위 중앙 0.771
```

**놓친 S 는 상대 RR 이 0.868 이다 — 13% 나 이른데도 놓쳤다.** 「조기성이 없어서 못 봤다」가
아니다. 조기성만으로는 **정상 박동의 변동과 겹쳐서** 못 가르는 것이다. 그리고 그 실패의
**90%가 AF 대리 상위**에 몰려 있다 — AF 에서는 RR 변동이 이소성보다 커서 조기성 축이 묻힌다.

**⇒ RR 축 안에서는 더 못 간다. 다른 시간 축이 필요하다.**

## ★★★ 사용자 임상 가설 — 그리고 대수적 정리

### (a) P-P interval 을 봐야 한다 → **그건 정확히 ΔPR 이다**

P 시각 = R 시각 − PR 이므로

```
PP(i) = P(i) − P(i−1) = RR(i) − [PR(i) − PR(i−1)] = RR(i) − ΔPR(i)
검증: max|PP − (RR − ΔPR)| = 0.0  (구성으로 정확)
```

**「P-P 가 R-R 과 어긋나는 그 지점」 = ΔPR ≠ 0 인 박동**이다. 특징 두 개가 하나로 압축되고,
**RR 이 이미 쓰는 정보와 직교**한다 — 이게 Q4-H 가 실패한 중복 문제를 구조적으로 피한다.
생리적 근거도 정확하다: 이소성 초점은 SA node 와 다른 자리라 **심방내 전도 경로·거리가
달라져 PR 이 바뀐다**.

### (b) P 폭 / QRS 폭 — 전도 속도 비 ⚠️ **정의를 고쳤다**

심방내 전도(P 폭)와 심실내 전도(QRS 폭)는 **다른 조직**이다. 이소성 심방 초점이면 P 폭은
늘지만 QRS 폭은 그대로 → **비가 오른다**. 비로 쓰면 개인별 진폭·필터 특성이 상쇄된다.
⚠️ 원안은 QRS 를 「Q 시작 − S **시작**」으로 잡았는데, 표준 QRS 폭은 **Q 시작 ~ S 종료
(J point)** 다. **Q 시작 ~ S 종료**로 고쳐 넣는다.

### (c) P 폭 / T 폭 — 넣되 **심박수 교란**을 안다

T 폭(재분극)은 **심박수 의존**이 크다(QT rate dependence). 그래서 (b)보다 약할 것으로
본다. 넣되 **RR 잔차화**를 같이 준비한다.

### (d) T 종료 ~ P 시작(TP 구간) ⚠️ **이대로 넣으면 Q4-H 를 반복한다**

```
RR = TP + P + PR + QRS + ST + T
```

나머지 구간이 안 변하면 **TP = RR − 상수**, 즉 **RR 의 재표현**이다. 합성 확인:

```
나머지 고정   ρ(TP, RR) = +1.0000 · RR 잔차 SD 3.1e-14  ← 남는 정보 **없음**
실제(잡음)    ρ(TP, RR) = +0.9766 · RR 잔차 SD 8.9      ← 남는 정보 **있음**
```

⇒ **TP 를 그냥 넣지 않고 `TP/RR` 과 `TP 의 RR 회귀 잔차`로 넣는다.** 잔차화는
**중복이면 0 을 남기고 새 정보가 있으면 그것만 남기는** 자동 안전장치다.

## ★★★ 이번에 새로 짓는 자 — **중복 감사**

Q4-H 가 죽은 이유는 「추가 특징이 이미 있는 축의 재표현」이었는데, **그걸 사후에야 알았다**.
이제 **모든 새 열을 base 9열에 회귀시켜 R² 를 먼저 낸다.** R² > 0.90 이면 중복으로
표시하고 그 열의 기여를 따로 읽는다. 이게 앞으로 모든 특징 추가의 **입구 검사**다.

## 딥러닝 절(선택 · GPU 필요)

사용자가 허가했으므로 **1D CNN 팔을 넣는다.** 단 정직하게:
- LORO 56 폴드로 CNN 을 56번 학습하는 건 과하다 → **레코드 그룹 5-겹**으로 하고
  **같은 5-겹에서 CPU 팔도 다시 재서** 같은 자로 비교한다(다른 자로 비교하면 무의미하다)
- 팔: `dl_wave`(파형만) · `dl_hybrid`(CNN 임베딩 + RR + 구간)
- **GPU 가 없으면 이 절은 건너뛴다**(명시적으로 로그에 남긴다). 관문 아님
- ⚠️ 문헌: 환자분리 SVEB 는 1D CNN 도 **74.56%** 로 20년째 ~75% 다. **큰 기대는 안 한다**

## 관문 (사전등록)

| 관문 | 무엇 | 통과 기준 |
|---|---|---|
| **O0** | 코호트 · 파형 · **델리네이션 자기검증**(PR 이 생리 범위인가) | 구성. 깨지면 **중단** |
| **O1 ★★★** | **중복 감사** — 새 열이 base 의 재표현인가 | 관문 아님 · **입구 검사** |
| **O2 ★★★ 주 관문** | `intv` vs `ishuf`(차원 동일) · **달성률@300**(1차) + AUROC(2차) | 측정된 raw 영점 상단 초과 |
| **O3 ★★** | `full`(RR+형태+구간) vs `fshuf` · Q4-J 형태 재현 | 배포·기전 병기 |
| **O4 ★★** | 열별 기여 · 사용자 가설 (a)~(d) 개별 판정 | 관문 아님 |
| **O5** | 딥러닝 5-겹(GPU 있을 때만) | 관문 아님 |
| **O6** | 필요표본 · 검산표 | R38 ⑦ · R39 ⑤ · R41 ② |

⚠️ **새 데이터 0** — `svdb_data5.npz` 의 `beat`·`sym`·`pre_rr` 파생만 쓴다.


In [ ]:
# CELL 0 — 공용 사전점검
import numpy as np

def decide(lo, hi, thr, direction):
    if direction not in (">", "<"):
        raise ValueError("direction 은 '>' 또는 '<'")
    if not (np.isfinite(lo) and np.isfinite(hi) and np.isfinite(thr)):
        return "⚠️ 미결"
    if direction == ">":
        if lo > thr: return "✅ 지지"
        if hi < thr: return "❌ 기각"
    else:
        if hi < thr: return "✅ 지지"
        if lo > thr: return "❌ 기각"
    return "⚠️ 미결"

def mde(lo, hi):
    return (hi - lo) / 2.0 if np.isfinite(lo) and np.isfinite(hi) else float("nan")

def boot_mean(v, seed, nb=3000, q=2.5):
    d = np.asarray(v, float); d = d[np.isfinite(d)]
    if len(d) < 3:
        return float("nan"), float("nan"), float("nan"), len(d)
    rng = np.random.RandomState(seed)
    b = [d[rng.randint(0, len(d), len(d))].mean() for _ in range(nb)]
    return (float(d.mean()), float(np.percentile(b, q)),
            float(np.percentile(b, 100 - q)), len(d))

def boot_pair(a, b, seed, nb=3000, q=2.5):
    a = np.asarray(a, float); b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b); a, b = a[m], b[m]
    if len(a) < 3:
        return float("nan"), float("nan"), float("nan"), len(a)
    rng = np.random.RandomState(seed)
    d = [(b[j] - a[j]).mean() for j in (rng.randint(0, len(a), len(a)) for _ in range(nb))]
    return (float((b - a).mean()), float(np.percentile(d, q)),
            float(np.percentile(d, 100 - q)), len(a))

def need_super(n, half, eff, p80=False):
    if not np.isfinite(half) or not np.isfinite(eff) or abs(eff) < 1e-9 or n < 1:
        return float("nan")
    r = float(n) * (half / abs(eff)) ** 2
    return r * 2.04 if p80 else r

class AssetError(RuntimeError): pass
print("CELL 0 ✅")


In [ ]:
# CELL 1 — 설정 · 사전등록
import os, sys, json, importlib, time, warnings
importlib.invalidate_caches(); warnings.filterwarnings("ignore")

SMOKE = os.environ.get("MEDKOS_SMOKE") == "1"
_ENV_ROOT = os.environ.get("MEDKOS_DRIVE_ROOT")
if _ENV_ROOT:
    DRIVE_ROOT = _ENV_ROOT
else:
    try:
        from google.colab import drive; drive.mount("/content/drive", force_remount=False)
        DRIVE_ROOT = "/content/drive/MyDrive"
    except Exception as e:
        print("⚠️ Colab 아님:", e); DRIVE_ROOT = "/content"
PROJECT = os.path.join(DRIVE_ROOT, "MedKOS", "ecg-model")
MITBIH  = os.path.join(DRIVE_ROOT, "mitbih")
sys.path.insert(0, os.path.join(PROJECT, "lib"))
from medkos_run import MedKOSRun

SEED0, IDX_S = 20260805, 1
RHY_K = (5, 10, 20, 32)
MIN_S, MIN_N = 25, 25
DEV_EVERY = 4
MAIN_K = 300
MAX_NEG_SLOPE, MIN_AUC_SLOPE = 0.10, 0.55
FS = 360.0                       # svdb_data5 는 128Hz → 360Hz 재표본
R_IDX = 100                      # 창 −278 ~ +556ms · R 피크 위치
# ── 델리네이션 창·임계 (전부 **사전 고정** · R34 ②)
W_QRS  = (85, 125)
W_P_S  = (20, 85)                # P 탐색 (−222 ~ −42ms)
W_T_S  = (135, 265)              # T 탐색 (+97 ~ +458ms)
W_Q_S  = (72, 145)               # QRS 탐색 (Q 시작 ~ S **종료**)
FRAC_QRS, FRAC_P, FRAC_T = 0.10, 0.25, 0.25
LAG = 30                         # P 정렬 탐색 ±83ms
P_CORR_MIN = 0.30                # 이보다 낮으면 **P 검출 실패**(AF 에서 흔하다)
TMPL_LO, TMPL_HI, TMPL_MIN = 0.92, 1.08, 30
DUP_R2 = 0.90                    # ★ 중복 감사 문턱
NB_BOOT = 400 if SMOKE else 2000
N_PERM  = 2   if SMOKE else 8
# ── 딥러닝 절(선택)
DL_FOLDS, DL_EPOCH, DL_BATCH, DL_EMB = 5, (1 if SMOKE else 4), 1024, 16

ARMS = ("base", "morph", "intv", "ishuf", "full", "fshuf")
MAIN_CT = "intv-ishuf"
CONTRASTS = (("morph-base",  "base",  "morph"),
             ("intv-base",   "base",  "intv"),
             ("intv-ishuf",  "ishuf", "intv"),
             ("full-morph",  "morph", "full"),
             ("full-fshuf",  "fshuf", "full"))
# ★★★ 사전등록 변경 — 주 지표를 **달성률@300**(1차) + 매크로 AUROC(2차) 공동으로
PRIMARY, SECONDARY = "ach", "auc"
READ_ORDER = ("O0", "O1", "O2", "O3", "O4", "O5", "O6")
SV5 = os.path.join(MITBIH, "svdb_data5.npz")

REF = dict(
    n_ok=56, mean_prev=0.0837,
    q4j=dict(base=dict(auc=0.9420, ap=0.5796, sens=0.7459, ppv=0.3620, ach=0.8074),
             morph=dict(auc=0.9529, ap=0.7236, sens=0.8211, ppv=0.4280, ach=0.9018),
             mshuf=dict(auc=0.9386, ap=0.5722, sens=0.7421, ppv=0.3612, ach=0.8044),
             monly=dict(auc=0.6199, ap=0.1694, sens=0.2978, ppv=0.1510, ach=0.3249),
             gate_auc=(0.0142, 0.0012, 0.0273), gate_thr=0.0248,
             fp_n=6338, fp_v=4380, fp_v_after=1444, fp_n_after=8166,
             fn_run=0.397, fn_hi=0.900, fn_rel=0.868, fn_pct=0.771,
             uni_p_energy=0.2361, uni_qrs_width=0.1150, uni_base_max=0.4337,
             moe=-0.0116),
    q7s=dict(ceiling=0.0213, need=222, pool=126),
    lit=[("de Chazal 2004 (IEEE TBME · DS2)", 0.759, 0.385),
         ("Llamedo & Martinez 2011 (IEEE TBME)", 0.77, 0.39),
         ("1D CNN inter-patient SVEB", 0.7456, None)])

RULE_CHECK = {
    "R11 매크로":       "환자 단위 · 상한과 함께 읽는다",
    "R16 fallback 없음": "`beat`·`sym` 없으면 **중단** · 델리네이션 자기검증 실패도 **중단**",
    "R22 누출 없음":     "LORO · 템플릿·델리네이션·잔차화 전부 **라벨을 안 쓴다**",
    "R26 영점":         "영점은 **raw(비교정)** · rep 수준 산포 병기",
    "R29 ② 분기 금지":   "O0 이 깨지면 아래를 **안 읽는다**",
    "R33 ① MDE":        "관문마다 MDE. **미결 ≠ 등가**. ★ **영점이 관측보다 흐리면 기각을 "
                        "미결로 강등**한다 — 보수적 문턱이 ❌ 를 만들면 안 된다",
    "R34 ② 문턱 금지":  "창·임계·예산·중복문턱을 **사전 고정**. 지표 변경도 **데이터 보기 전**",
    "R35 ① 자 먼저":    "★★★ **중복 감사(O1)가 자다** — Q4-H 를 사후에야 안 실수를 입구에서 잡는다",
    "R36 ② 선택 편의":  "열 선택을 안 한다 — 12열을 **통째로** 넣고 블록으로 판정한다",
    "R38 ⑦ 요약 정합":  "★★★ **내 지표 선택 오류를 명시**한다 — AUROC 는 상위 꼬리 개입을 못 본다",
    "R39 ① 대안설명":   "P 검출 실패율이 AF 에서 높으면 그 자체가 **레코드 수준 신호**다 — "
                        "레코드 내 순위엔 안 쓰인다는 걸 안다",
    "R40 ① 가시성≠판별": "Q7-S′ 가 P 에서 겪었다 — 구간이 **판별로 전환되는지**만 본다",
    "R41 ② 0 근처":     "효과가 0 근처면 필요표본은 해석 불가",
}

CONFIG = dict(
    exp="quest46_q4k_pp_interval", quest="ailab-2026-0046", step="pp-interval",
    parent_exp=["quest46_q4j_error_anatomy"],
    purpose=("★★★ **Q4-J 는 성공했는데 내가 잘못된 자로 쟀다.** 형태 8열이 달성률을 "
             "0.8074 → **0.9018**(+0.0944 · +11.7%) · PR-AUC 0.5796 → **0.7236**(+24.8%) · "
             "PPV 0.3620 → **0.4280** 로 올렸는데, 내가 주 관문으로 건 **매크로 AUROC 는 "
             "+0.0109** 밖에 안 움직여 ⚠️ 미결이 났다. 이유는 명확하다 — 형태는 **상위 300개에서 "
             "V 를 걷어내는** 일을 한다(V 위양성 **4380 → 1444 · 67% 감소**, N 위양성은 6338 → "
             "8166 으로 **빈자리를 채웠다**). 그건 **순위 맨 위**에서 벌어지는 일이고 AUROC 는 "
             "전체 순위쌍에 지배된다. **Q4-G 의 ρ(AUROC, 달성률)=+0.8869 는 「레코드 간 상관」이지 "
             "「개입이 둘을 같이 움직인다」가 아닌데 내가 혼동했다.** ⇒ **사전등록 변경(데이터 "
             "보기 전)**: 주 지표를 **달성률@300**(1차) + 매크로 AUROC(2차) **공동**으로 바꾼다. "
             "★★★ **그리고 N1 오류 해부가 다음 축을 지목했다**: 놓친 S 의 **상대 RR 중앙이 "
             "0.868** 이다 — **13% 나 이른데도 놓쳤다**. 조기성이 없어서가 아니라 **조기성만으로는 "
             "정상 변동과 겹쳐서** 못 가른 것이고, 그 실패의 **90%가 AF 대리 상위**에 몰려 있다. "
             "⇒ **RR 축 안에서는 더 못 간다.** ★★★ **그래서 사용자 임상 가설을 수치화한다**: "
             "(a) **P-P interval** — 대수적으로 `PP(i) = RR(i) − ΔPR(i)` 이므로 「P-P 가 R-R 과 "
             "어긋나는 지점」은 **정확히 ΔPR ≠ 0** 이다(검증 max|Δ| = 0.0). 이소성 초점은 SA node "
             "와 다른 자리라 **심방내 전도 경로가 달라져 PR 이 바뀐다** — RR 과 **직교**하는 새 축이다 "
             "(b) **P 폭 / QRS 폭** — 심방내 전도와 심실내 전도는 다른 조직이라 이소성이면 비가 "
             "오른다. ⚠️ 원안의 QRS 정의(Q 시작−S **시작**)를 **표준(Q 시작~S 종료)** 으로 고쳤다 "
             "(c) **P 폭 / T 폭** — T 는 심박수 의존이 커 (b)보다 약할 것이다 "
             "(d) **TP 구간** — ⚠️ **이대로 넣으면 Q4-H 를 반복한다**: `RR = TP+P+PR+QRS+ST+T` "
             "라 나머지가 고정이면 **TP = RR − 상수**(합성 확인 ρ=+1.0000 · 잔차 SD 3.1e-14). "
             "그래서 **`TP/RR` 과 `TP 의 RR 회귀 잔차`** 로 넣는다 — 잔차화는 **중복이면 0 을, "
             "새 정보가 있으면 그것만** 남기는 자동 안전장치다. ★★★ **새 자 하나를 짓는다 — "
             "중복 감사**: 모든 새 열을 base 9열에 회귀시켜 R² 를 **먼저** 낸다(문턱 0.90). "
             "Q4-H 를 사후에야 안 실수를 **입구에서** 잡는다."),
    dataset="SVDB — svdb_data5.npz (새 데이터 0 · beat·sym·pre_rr 파생만)",
    arms=list(ARMS), main_contrast=MAIN_CT, primary=PRIMARY, secondary=SECONDARY,
    main_k=MAIN_K, fs=FS, r_idx=R_IDX,
    windows=dict(qrs=W_QRS, p_search=W_P_S, t_search=W_T_S, q_search=W_Q_S,
                 frac=(FRAC_QRS, FRAC_P, FRAC_T), lag=LAG, p_corr_min=P_CORR_MIN),
    dup_r2=DUP_R2, read_order=READ_ORDER, dev_every=DEV_EVERY, n_boot=NB_BOOT,
    n_perm=N_PERM, dl=dict(folds=DL_FOLDS, epoch=DL_EPOCH, batch=DL_BATCH, emb=DL_EMB),
    smoke=SMOKE, ref=REF, rule_check=RULE_CHECK,
    predictions={
        "O0": "코호트 · 파형 · **델리네이션 자기검증**(PR 중앙이 생리 범위 80~220ms 인가 · "
              "P 검출률). 깨지면 **중단**",
        "O1": "★★★ **중복 감사(입구 검사)** — 새 12열 각각을 base 9열에 회귀시켜 R². "
              "0.90 초과면 **중복**으로 표시한다. Q4-H 의 `pre/base` 가 여기서 걸렸어야 했다",
        "O2": "★★★ **주 관문** — `intv`(base 9 + 구간 12) vs `ishuf`(차원 동일). "
              "**달성률@300 이 1차**, 매크로 AUROC 가 2차",
        "O3": "★★ `full`(RR+형태+구간) vs `fshuf` · Q4-J 의 형태 결과를 같은 런에서 재현",
        "O4": "★★ **열별 기여와 사용자 가설 개별 판정** — (a) ΔPR·PP/RR (b) P폭/QRS폭 "
              "(c) P폭/T폭 (d) TP/RR·TP잔차. 각 블록을 빼고 넣어 본다",
        "O5": "딥러닝 5-겹(**GPU 있을 때만**) — `dl_wave`·`dl_hybrid` vs 같은 겹의 CPU 팔. "
              "문헌상 환자분리 SVEB 는 1D CNN 도 74.56% 라 **큰 기대는 안 한다**",
        "O6": "필요표본 · 결론 검산표"},
    caveat=("★★★ **내 지표 선택 오류를 명시한다**(R38 ⑦) — Q4-J 에서 개입은 **상위 꼬리**에 "
            "작용했는데 나는 **전체 순위 지표**로 관문을 걸었다. 세 런 연속으로 내 판단이 "
            "틀렸고, 이번 것은 **대조군이 아니라 지표**의 문제였다. "
            "★★★ **사용자 가설 (d)는 그대로 넣으면 Q4-H 의 반복**이다 — TP 는 RR 의 재표현이 "
            "될 수 있어 **RR 잔차화**로만 넣는다. (b)는 **QRS 폭 정의를 표준으로 고쳤다**. "
            "★★ **P 검출 실패는 AF 신호지만 레코드 내 순위엔 거의 안 쓰인다**(R39 ①) — "
            "레코드 안에서 거의 상수면 within-record 순위를 못 바꾼다. 그래서 `p_found` 를 "
            "넣되 **그 열의 중복 감사와 레코드 내 분산**을 같이 본다. "
            "★★ **Q7-S′ 의 P 갈래 재개가 아니다**(상한 +0.0213 · 필요 222명) — 그건 **P 형태**의 "
            "판별력이었고, 이번은 **P 의 시간(PR·PP·구간비)** 이다. 다른 양이다. "
            "★ **DL 은 관문이 아니다** — GPU 없으면 건너뛰고, 있어도 **같은 5-겹에서 CPU 팔을 "
            "다시 재서** 비교한다(다른 자로 비교하면 무의미하다)."))
np.random.seed(SEED0)
run = MedKOSRun("quest46_q4k_pp_interval", CONFIG, project=PROJECT)
run.log("설정 ✅ **Q4-K — P-P / PR / 구간비: 이소성 초점을 시간으로 잡는다**")
run.log(f"  ★★★ **지표 오류 정정** — Q4-J 형태는 달성률 {REF['q4j']['base']['ach']} → "
        f"**{REF['q4j']['morph']['ach']}**(+0.0944) 인데 AUROC 는 +0.0109 뿐이었다. "
        f"주 지표를 **달성률@300**(1차)로 바꾼다")
run.log(f"  ★★★ **놓친 S 의 상대 RR 중앙 {REF['q4j']['fn_rel']}** — 이른데도 놓쳤다. "
        f"RR 축 안에서는 더 못 간다")
run.log(f"  ★★★ **PP(i) = RR(i) − ΔPR(i)** — 「P-P 가 R-R 과 어긋나는 지점」은 **정확히 ΔPR** 이다")
run.log(f"  ⚠️ TP 구간은 **RR 잔차화**로만 넣는다 — 안 그러면 Q4-H 의 중복을 반복한다")
if SMOKE:
    run.log(f"  ⚠️ **스모크런** — 비용 손잡이만 축소(NB_BOOT={NB_BOOT} · N_PERM={N_PERM})")
run.log("\n  사전등록 규칙 체크리스트 (R29 ③)")
for k_, v_ in RULE_CHECK.items():
    run.log(f"    [x] {k_:<18} {v_}")


In [ ]:
# CELL 2 — 【O-0】 코호트 · RR 9열 · 형태 8열(Q4-J) · ★★★ 구간 12열(새것)
import pandas as pd
from collections import Counter
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import average_precision_score, roc_auc_score

run.log("\n" + "=" * 100)
run.log("【O-0】 코호트 · 델리네이션 · ★★★ 구간 12열")
run.log("=" * 100)
VERD, NOTE = {}, {}
def g_(k, v, d):
    VERD[k] = v; NOTE[k] = d; run.log(f"  {k:<5}{v}  {d}")

if not os.path.exists(SV5):
    raise AssetError(f"{SV5} 없음(R16)")
D5 = np.load(SV5, allow_pickle=True)
for need in ("pid", "y3", "pre_rr", "post_rr", "beat", "sym"):
    if need not in D5.files:
        raise AssetError(f"`{need}` 가 자산에 없다(R16)")
PID = np.asarray(D5["pid"]).astype(int); Y3 = np.asarray(D5["y3"]).astype(int)
PRE = np.asarray(D5["pre_rr"], float); POST = np.asarray(D5["post_rr"], float)
K = np.where(Y3 >= 0)[0]
RID = PID[K]; TT_ = (Y3[K] == IDX_S)
pre = PRE[K].astype(float); post = POST[K].astype(float)
BEAT = np.asarray(np.asarray(D5["beat"])[K], dtype="float32")
SYM = np.asarray(D5["sym"]).astype("<U2")[K]
NL, LW = BEAT.shape[1], BEAT.shape[2]
if LW <= W_T_S[1]:
    raise AssetError(f"파형 길이 {LW} 가 사전 고정 T 창 {W_T_S} 보다 짧다(R34 ②)")
RS = np.array(sorted(set(RID.tolist())))
IDX_ALL = {int(r): np.where(RID == r)[0] for r in RS}
run.log(f"  파형 {BEAT.shape} · 리드 {NL} · 길이 {LW}")

_S = pd.Series(pre); _G = _S.groupby(pd.Series(RID))
def local_base(k):
    r = np.asarray(_G.apply(lambda x: x.shift(1).rolling(k, min_periods=1).median())).astype(float)
    return np.where(np.isfinite(r), r, pre)
_med = _G.transform("median").to_numpy()
_std = _G.transform("std").to_numpy(); _mean = _G.transform("mean").to_numpy()
BASE12 = local_base(12); REL = pre / (BASE12 + 1e-9)
F_BASE = np.nan_to_num(np.c_[_med - pre,
                             np.column_stack([1.0 - pre / (local_base(k) + 1e-9) for k in RHY_K]),
                             post - pre, np.nan_to_num(_std / (_mean + 1e-9)),
                             np.log1p(np.clip(pre, 0, None)), np.log1p(np.clip(post, 0, None))],
                       nan=0.0, posinf=0.0, neginf=0.0)

def corr_to(x, t):
    xc = x - x.mean(-1, keepdims=True); tc = t - t.mean(-1, keepdims=True)
    return (xc * tc).sum(-1) / (np.sqrt((xc ** 2).sum(-1) * (tc ** 2).sum(-1)) + 1e-9)

def span_1d(x, lo, hi, frac, base):
    """[lo,hi) 에서 |x−base| 최대점을 찾고, 그 봉우리의 frac 교차로 시작·끝을 낸다"""
    seg = np.abs(x[lo:hi] - base)
    k = int(np.argmax(seg)); amp = float(seg[k])
    if amp <= 1e-9:
        return lo, hi - 1, lo + k, 0.0
    m = seg > frac * amp
    i = k
    while i > 0 and m[i - 1]: i -= 1
    j = k
    while j < len(m) - 1 and m[j + 1]: j += 1
    return lo + i, lo + j, lo + k, amp

def width_batch(B, lo, hi, frac, base):
    """(n, 리드, L) 에서 창 [lo,hi) 봉우리 폭(표본)과 진폭 — 리드 평균"""
    seg = np.abs(B[:, :, lo:hi] - base[:, :, None])
    amp = seg.max(-1)                                   # (n, 리드)
    m = seg > frac * amp[..., None]
    first = m.argmax(-1)
    last = m.shape[-1] - 1 - m[:, :, ::-1].argmax(-1)
    w = np.where(m.any(-1), last - first + 1, 0)
    return w.mean(1).astype(float), amp.mean(1).astype(float), (first.mean(1) + lo)

INAMES = ["pr_rel", "dpr_ms", "pr_dev_ms", "pp_over_rr", "pw_over_qrsw", "pw_over_tw",
          "tp_over_rr", "tp_resid", "pw_rel", "p_corr", "p_amp_rel", "p_found"]
DELIN = {}
def interval_feats():
    out = np.zeros((len(K), 12), float)
    for r in RS:
        ii = IDX_ALL[int(r)]
        okm = (REL[ii] >= TMPL_LO) & (REL[ii] <= TMPL_HI)
        if int(okm.sum()) < TMPL_MIN: okm = np.ones(len(ii), bool)
        B = BEAT[ii]
        T = np.median(B[okm], axis=0)                    # 무라벨 템플릿 (리드, L)
        lead = int(np.argmax(np.ptp(T[:, W_QRS[0]:W_QRS[1]], axis=-1)))
        x = T[lead].astype(float)
        iso = float(np.median(x[W_P_S[0]:W_P_S[0] + 8]))  # 등전위 대리
        p_on, p_off, p_pk, p_amp = span_1d(x, *W_P_S, FRAC_P, iso)
        t_on, t_off, t_pk, t_amp = span_1d(x, *W_T_S, FRAC_T, iso)
        q_on, s_off, r_pk, q_amp = span_1d(x, *W_Q_S, FRAC_QRS, iso)
        DELIN[int(r)] = dict(lead=lead, p=(p_on, p_off, p_pk), t=(t_on, t_off, t_pk),
                             q=(q_on, s_off, r_pk))
        # ── 박동별 P 정렬(템플릿 P 조각과의 교차상관)
        pw0 = max(2, p_off - p_on + 1)
        tp_seg = T[:, p_on:p_on + pw0]
        best_c = np.full(len(ii), -2.0); best_l = np.zeros(len(ii), int)
        for lag in range(-LAG, LAG + 1):
            a, b = p_on + lag, p_on + lag + pw0
            if a < 0 or b > LW: continue
            c = corr_to(B[:, :, a:b], tp_seg).mean(1)
            upd = c > best_c
            best_c[upd] = c[upd]; best_l[upd] = lag
        p_found = (best_c >= P_CORR_MIN).astype(float)
        p_pk_b = p_pk + best_l                            # 박동별 P 봉우리 위치
        p_on_b = p_on + best_l
        base_b = np.median(B[:, :, W_P_S[0]:W_P_S[0] + 8], axis=-1)   # (n, 리드)
        # ── 폭들
        pw, pa, _ = width_batch(B, max(0, p_on - LAG), min(LW, p_off + LAG + 1), FRAC_P, base_b)
        qw, qa, _ = width_batch(B, W_Q_S[0], W_Q_S[1], FRAC_QRS, base_b)
        tw, ta, _ = width_batch(B, W_T_S[0], W_T_S[1], FRAC_T, base_b)
        # ── 구간(표본 → ms)
        pr = (R_IDX - p_pk_b).astype(float)                       # 표본
        pr_ms = pr / FS * 1000.0
        med_pr = float(np.median(pr_ms[p_found > 0])) if (p_found > 0).any() else float(np.median(pr_ms))
        dpr = np.r_[0.0, np.diff(pr_ms)]
        rr_s = pre[ii] * FS
        t_off_b = np.full(len(ii), float(t_off))                  # T 는 QRS 를 따라간다(재정렬 안 함)
        tp = np.r_[np.nan, rr_s[1:] + p_on_b[1:] - t_off_b[:-1]]
        tp = np.where(np.isfinite(tp), tp, np.nanmedian(tp))
        tp_rr = tp / (rr_s + 1e-9)
        # ★ TP 의 RR 회귀 잔차 — 레코드 안에서, **라벨 없이**
        A = np.c_[np.ones(len(ii)), rr_s]
        coef, *_ = np.linalg.lstsq(A, tp, rcond=None)
        tp_res = tp - A @ coef
        sd = float(np.std(tp_res)) + 1e-9
        med_pw = float(np.median(pw)) + 1e-9; med_pa = float(np.median(pa)) + 1e-9
        out[ii, 0] = pr_ms / (med_pr + 1e-9)
        out[ii, 1] = dpr
        out[ii, 2] = pr_ms - med_pr
        out[ii, 3] = 1.0 - (dpr / 1000.0 * FS) / (rr_s + 1e-9)
        out[ii, 4] = pw / (qw + 1e-9)
        out[ii, 5] = pw / (tw + 1e-9)
        out[ii, 6] = tp_rr
        out[ii, 7] = tp_res / sd
        out[ii, 8] = pw / med_pw
        out[ii, 9] = best_c
        out[ii, 10] = pa / med_pa
        out[ii, 11] = p_found
    return np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0)

def morph_feats():
    out = np.zeros((len(K), 8), float)
    WQ, WF, WS, WP, WW = (85, 125), (60, 220), (130, 260), (25, 75), (80, 130)
    for r in RS:
        ii = IDX_ALL[int(r)]
        okm = (REL[ii] >= TMPL_LO) & (REL[ii] <= TMPL_HI)
        if int(okm.sum()) < TMPL_MIN: okm = np.ones(len(ii), bool)
        B = BEAT[ii]; T = np.median(B[okm], axis=0)
        cq = corr_to(B[:, :, WQ[0]:WQ[1]], T[:, WQ[0]:WQ[1]])
        cf = corr_to(B[:, :, WF[0]:WF[1]], T[:, WF[0]:WF[1]])
        cs = corr_to(B[:, :, WS[0]:WS[1]], T[:, WS[0]:WS[1]])
        seg = B[:, :, WW[0]:WW[1]]; med = np.median(seg, axis=-1, keepdims=True)
        amp = np.abs(seg - med).max(-1, keepdims=True) + 1e-9
        wid = (np.abs(seg - med) > 0.5 * amp).mean(-1)
        q = B[:, :, WQ[0]:WQ[1]]; ptp = q.max(-1) - q.min(-1)
        tq = T[:, WQ[0]:WQ[1]]; tptp = float(np.mean(tq.max(-1) - tq.min(-1))) + 1e-9
        area = np.abs(q - np.median(q, axis=-1, keepdims=True)).sum(-1)
        tarea = float(np.mean(np.abs(tq - np.median(tq, axis=-1, keepdims=True)).sum(-1))) + 1e-9
        p = B[:, :, WP[0]:WP[1]]
        pe = np.sqrt(((p - p.mean(-1, keepdims=True)) ** 2).mean(-1))
        tp_ = T[:, WP[0]:WP[1]]
        tpe = float(np.mean(np.sqrt(((tp_ - tp_.mean(-1, keepdims=True)) ** 2).mean(-1)))) + 1e-9
        out[ii, 0] = cq.min(1); out[ii, 1] = cq.mean(1); out[ii, 2] = cf.min(1)
        out[ii, 3] = cs.min(1); out[ii, 4] = wid.mean(1)
        out[ii, 5] = ptp.mean(1) / tptp; out[ii, 6] = area.mean(1) / tarea
        out[ii, 7] = pe.mean(1) / tpe
    return np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0)

T_FEAT = time.time()
MORPH = morph_feats(); INTV = interval_feats()
run.log(f"  ({time.time()-T_FEAT:.0f}초) 형태 8열 · 구간 12열 — " + " · ".join(INAMES))

# ── ★ O0 델리네이션 자기검증 (생리 범위 · R16)
pr_ms_all = INTV[:, 2] + 0.0
found = INTV[:, 11]
PR_MED = float(np.median([np.median((R_IDX - (DELIN[int(r)]["p"][2])) / FS * 1000.0)
                          for r in RS]))
FOUND_RATE = float(found.mean())
run.log(f"  ★ 델리네이션 자기검증 — 템플릿 PR 중앙 **{PR_MED:.1f}ms** · P 검출률 "
        f"**{FOUND_RATE:.1%}** · P 폭 중앙 "
        f"{np.median([(DELIN[int(r)]['p'][1]-DELIN[int(r)]['p'][0])/FS*1000 for r in RS]):.1f}ms · "
        f"QRS 폭 중앙 "
        f"{np.median([(DELIN[int(r)]['q'][1]-DELIN[int(r)]['q'][0])/FS*1000 for r in RS]):.1f}ms")
if not (60.0 <= PR_MED <= 260.0):
    raise AssetError(f"O0 실패 — 템플릿 PR 중앙 {PR_MED:.1f}ms 가 생리 범위(60~260) 밖이다(R16)")
if FOUND_RATE < 0.20:
    raise AssetError(f"O0 실패 — P 검출률 {FOUND_RATE:.1%} 가 너무 낮다(R16)")

_rs = np.random.RandomState(SEED0 + 7)
def rec_shuffle(M):
    S_ = M.copy()
    for r in RS:
        ii = IDX_ALL[int(r)]; S_[ii] = M[ii][_rs.permutation(len(ii))]
    return S_
I_SH = rec_shuffle(INTV); F_SH = rec_shuffle(np.c_[MORPH, INTV])
_moved = float(np.mean(np.any(np.abs(I_SH - INTV) > 1e-12, axis=1)))
if _moved < 0.5:
    raise AssetError(f"차원 대조군이 거의 항등이다({_moved:.3f})(R35 ①)")

FEAT = {"base": F_BASE, "morph": np.c_[F_BASE, MORPH],
        "intv": np.c_[F_BASE, INTV], "ishuf": np.c_[F_BASE, I_SH],
        "full": np.c_[F_BASE, MORPH, INTV], "fshuf": np.c_[F_BASE, F_SH]}
for a, b in (("intv", "ishuf"), ("full", "fshuf")):
    if FEAT[a].shape[1] != FEAT[b].shape[1]:
        raise AssetError(f"차원 대조군 불일치 — {a} {FEAT[a].shape[1]} vs {b} {FEAT[b].shape[1]}")
run.log("  차원 — " + " · ".join(f"{a} {FEAT[a].shape[1]}" for a in ARMS)
        + f" · 대조군 이동 {_moved:.1%}")

IDXS = {int(r): IDX_ALL[int(r)] for r in RS}
REC_OK = [int(r) for r in RS
          if TT_[IDXS[int(r)]].sum() >= MIN_S and (~TT_[IDXS[int(r)]]).sum() >= MIN_N]
BURD = {r: float(TT_[IDXS[r]].mean()) for r in REC_OK}
NS_ = {r: int(TT_[IDXS[r]].sum()) for r in REC_OK}
NRE = len(REC_OK)
run.log(f"  레코드 {len(RS)} · 채점 가능 **{NRE}** · 평균 유병률 "
        f"{np.mean([BURD[r] for r in REC_OK]):.4f}")

SLOPES, SLOPE_BY, _CUR = [], {}, [None]
def make_cal(s, y):
    lr = LogisticRegression(max_iter=3000, C=1e6).fit(np.asarray(s).reshape(-1, 1),
                                                       np.asarray(y).astype(int))
    a, b = float(lr.coef_[0, 0]), float(lr.intercept_[0])
    SLOPES.append(a)
    if _CUR[0] is not None: SLOPE_BY.setdefault(_CUR[0], []).append(a)
    return lambda v: a * np.asarray(v, float) + b

def split_rest(held):
    rest = sorted([r for r in REC_OK if r != held], key=lambda r: (BURD[r], r))
    dv = [r for i, r in enumerate(rest) if i % DEV_EVERY == 0]
    return [r for r in rest if r not in set(dv)], dv

def fit_fold(X, held, y_override):
    tr_r, dv_r = split_rest(held)
    tr = np.concatenate([IDXS[r] for r in tr_r]); dv = np.concatenate([IDXS[r] for r in dv_r])
    te = IDXS[held]
    Ftr = X[tr]; fmu = Ftr.mean(0); fsd = Ftr.std(0) + 1e-9
    ytr = TT_[tr].astype(int) if y_override is None else np.asarray(y_override[held], int)
    lr = LogisticRegression(max_iter=3000, C=1.0).fit((Ftr - fmu) / fsd, ytr)
    f = lambda ii: lr.decision_function((X[ii] - fmu) / fsd)
    raw = f(te)
    return te, make_cal(f(dv), TT_[dv])(raw), raw

def loro(X, y_override=None, tag=None):
    out = np.full(len(K), np.nan); raw = np.full(len(K), np.nan); _CUR[0] = tag
    for held in REC_OK:
        te, v, rv = fit_fold(X, held, y_override)
        out[te] = v; raw[te] = rv
    return out, raw

def per_auc(L, recs=None):
    return {r: float(roc_auc_score(TT_[IDXS[r]].astype(int), L[IDXS[r]]))
            for r in (recs if recs is not None else REC_OK)}
def per_ap(L):
    return {r: float(average_precision_score(TT_[IDXS[r]].astype(int), L[IDXS[r]]))
            for r in REC_OK}
def at_k(L, r, k):
    idx = IDXS[r]; sc = L[idx]; yy = TT_[idx]
    k = int(min(max(1, k), len(idx)))
    fl = sc >= np.partition(sc, -k)[-k]
    tp = int((fl & yy).sum()); ceil = min(1.0, k / max(1, NS_[r]))
    return dict(sens=tp / max(1, NS_[r]), ppv=tp / max(1, int(fl.sum())), ceil=ceil,
                ach=(tp / max(1, NS_[r])) / ceil if ceil > 0 else np.nan, flag=fl)
def per_ach(L, recs=None):
    return {r: at_k(L, r, MAIN_K)["ach"] for r in (recs if recs is not None else REC_OK)}
CONFIG["cohort"] = dict(n_ok=NRE, moved=_moved, pr_med_ms=PR_MED, p_found=FOUND_RATE,
                        dims={a: int(FEAT[a].shape[1]) for a in ARMS})
run.save_json("config", CONFIG)


In [ ]:
# CELL 3 — 【O-A】 ★★★ O1 중복 감사(입구 검사) · 실행 · O0
run.log("\n" + "=" * 100)
run.log("【O-A】 ★★★ **O1 중복 감사** — 새 열이 base 의 재표현인가(Q4-H 를 입구에서 잡는다)")
run.log("=" * 100)
Zb = (F_BASE - F_BASE.mean(0)) / (F_BASE.std(0) + 1e-9)
def dup_r2(col):
    y = np.asarray(col, float)
    if np.std(y) < 1e-12: return 1.0
    lr = LinearRegression().fit(Zb, y)
    return float(max(0.0, min(1.0, lr.score(Zb, y))))
def within_var(col):
    """레코드 내 표준편차 / 전체 표준편차 — 0 이면 레코드 상수(순위를 못 바꾼다)"""
    y = np.asarray(col, float); tot = np.std(y) + 1e-12
    w = np.mean([np.std(y[IDXS[r]]) for r in REC_OK])
    return float(w / tot)
MN = ["corr_qrs_min", "corr_qrs_mean", "corr_full_min", "corr_st_min",
      "qrs_width", "amp_ratio", "area_ratio", "p_energy_ratio"]
run.log(f"  {'열':<16}{'블록':>8}{'R²(base)':>11}{'레코드내분산비':>14}{'|AUROC−.5|':>12}{'판정':>10}")
DUP = {}
def uni_one(col):
    a_ = []
    for r in REC_OK:
        ii = IDXS[r]; yy = TT_[ii].astype(int); v = col[ii]
        if 0 < yy.sum() < len(ii) and np.std(v) > 0:
            a_.append(abs(roc_auc_score(yy, v) - 0.5))
    return float(np.mean(a_)) if a_ else 0.0
for blk, M, names in (("형태", MORPH, MN), ("구간", INTV, INAMES)):
    for j, nm in enumerate(names):
        r2 = dup_r2(M[:, j]); wv = within_var(M[:, j]); u = uni_one(M[:, j])
        verdict = "⛔ 중복" if r2 > DUP_R2 else ("⚠️ 레코드상수" if wv < 0.15 else "✅ 새 축")
        DUP[nm] = dict(block=blk, r2=r2, within=wv, uni=u, verdict=verdict)
        run.log(f"  {nm:<16}{blk:>8}{r2:>11.4f}{wv:>14.3f}{u:>12.4f}{verdict:>10}")
n_dup = sum(1 for v in DUP.values() if v["verdict"].startswith("⛔"))
n_const = sum(1 for v in DUP.values() if v["verdict"].startswith("⚠️"))
run.log(f"  ⇒ 중복 **{n_dup}** · 레코드상수 **{n_const}** · 새 축 {len(DUP)-n_dup-n_const} "
        f"(문턱 R² > {DUP_R2})")
run.log(f"  ▸ Q4-H 의 `pre/base` 는 여기서 **⛔ 중복**으로 걸렸어야 했다 — 그게 이 자의 목적이다")
g_("O1", "(입구 검사)",
   f"중복 {n_dup} · 레코드상수 {n_const} · 새 축 {len(DUP)-n_dup-n_const} · "
   f"구간 블록 최고 단변량 {max(DUP[n]['uni'] for n in INAMES):.4f} "
   f"(형태 최고 {max(DUP[n]['uni'] for n in MN):.4f} · Q4-J 기존 RR 최고 "
   f"{REF['q4j']['uni_base_max']:.4f})")

run.log("\n" + "=" * 100)
run.log("【O-B】 실행 · O0")
run.log("=" * 100)
T0 = time.time()
L, LRAW = {}, {}
for a in ARMS:
    L[a], LRAW[a] = loro(FEAT[a], None, tag=a)
    run.log(f"  ({time.time()-T0:>5.0f}초) {a} 완료")
AUC = {a: per_auc(L[a]) for a in ARMS}
AUC_RAW = {a: per_auc(LRAW[a]) for a in ARMS}
AP = {a: per_ap(L[a]) for a in ARMS}
R300 = {a: {r: at_k(L[a], r, MAIN_K) for r in REC_OK} for a in ARMS}
ACH = {a: {r: R300[a][r]["ach"] for r in REC_OK} for a in ARMS}
CAL_GAP = max(abs(AUC[a][r] - AUC_RAW[a][r]) for a in ARMS for r in REC_OK)

bad_arms, skipped = [], []
for a in ARMS:
    sa = np.array(SLOPE_BY.get(a, []), float); mac = float(np.mean(list(AUC[a].values())))
    if len(sa) == 0: continue
    negf = float(np.mean(sa <= 0))
    if mac <= MIN_AUC_SLOPE: skipped.append((a, mac)); continue
    if np.median(sa) <= 0 or negf > MAX_NEG_SLOPE: bad_arms.append((a, float(np.median(sa)), negf))
if bad_arms:
    raise AssetError("O0 실패 — 체계적 반전: "
                     + " · ".join(f"{a} {m:+.4f}/{f:.1%}" for a, m, f in bad_arms) + "(R29 ②)")
g_("O0", "✅ 지지", f"PR 중앙 {PR_MED:.1f}ms · P 검출률 {FOUND_RATE:.1%} · 교정 전후 AUROC "
                   f"최대차 {CAL_GAP:.2e} · 팔별 기울기 통과 {len(ARMS)-len(skipped)}/{len(ARMS)}")

run.log(f"\n  {'팔':<8}{'차원':>6}{'달성률@300★':>14}{'매크로 AUROC':>15}{'PR-AUC':>10}"
        f"{'민감도@300':>12}{'PPV@300':>10}")
for a in ARMS:
    run.log(f"  {a:<8}{FEAT[a].shape[1]:>6}{np.mean(list(ACH[a].values())):>14.4f}"
            f"{np.mean(list(AUC[a].values())):>15.4f}{np.mean(list(AP[a].values())):>10.4f}"
            f"{np.mean([R300[a][r]['sens'] for r in REC_OK]):>12.4f}"
            f"{np.mean([R300[a][r]['ppv'] for r in REC_OK]):>10.4f}")
run.log(f"  (Q4-J 앵커 — base 달성률 {REF['q4j']['base']['ach']} · morph "
        f"{REF['q4j']['morph']['ach']} · mshuf {REF['q4j']['mshuf']['ach']})")
CONFIG["O0"] = dict(cal_gap=float(CAL_GAP), pr_med=PR_MED, p_found=FOUND_RATE,
                    ach={a: float(np.mean(list(ACH[a].values()))) for a in ARMS},
                    auc={a: float(np.mean(list(AUC[a].values()))) for a in ARMS},
                    ap={a: float(np.mean(list(AP[a].values()))) for a in ARMS},
                    sens={a: float(np.mean([R300[a][r]["sens"] for r in REC_OK])) for a in ARMS},
                    ppv={a: float(np.mean([R300[a][r]["ppv"] for r in REC_OK])) for a in ARMS})
CONFIG["O1"] = DUP
run.save_json("config", CONFIG)


In [ ]:
# CELL 4 — 【O-C】 영점(공동 주 지표) · ★★★ O2 주 관문 · O3
run.log("\n" + "=" * 100)
run.log("【O-C】 영점 · ★★★ **O2 — 구간 12열이 달성률을 올리는가**(차원 동일 대조)")
run.log("=" * 100)
run.log(f"  ★ 영점은 **raw(비교정)** · **달성률과 AUROC 둘 다** 잰다 (reps={N_PERM})")
NUL = {c[0]: {"ach": {r: [] for r in REC_OK}, "auc": {r: [] for r in REC_OK}}
       for c in CONTRASTS}
REPM = {c[0]: {"ach": [], "auc": []} for c in CONTRASTS}
for s_ in range(N_PERM):
    rr = np.random.RandomState(SEED0 + 400 + s_)
    yov = {}
    for held in REC_OK:
        tr_r, _ = split_rest(held)
        tr = np.concatenate([IDXS[r] for r in tr_r])
        yov[held] = TT_[tr].astype(int)[rr.permutation(len(tr))]
    RW = {a: loro(FEAT[a], yov)[1] for a in ARMS}
    Aa = {a: per_auc(RW[a]) for a in ARMS}; Ah = {a: per_ach(RW[a]) for a in ARMS}
    for nm, x_, y_ in CONTRASTS:
        for key, src in (("auc", Aa), ("ach", Ah)):
            d = [src[y_][r] - src[x_][r] for r in REC_OK]
            REPM[nm][key].append(float(np.mean(d)))
            for i_, r in enumerate(REC_OK): NUL[nm][key][r].append(d[i_])
    run.log(f"    ({time.time()-T0:>5.0f}초) 영점 rep {s_+1}/{N_PERM}")

NSTAT, NREP = {}, {}
for nm, x_, y_ in CONTRASTS:
    NSTAT[nm], NREP[nm] = {}, {}
    for key in ("ach", "auc"):
        NSTAT[nm][key] = boot_mean([float(np.mean(v)) for v in NUL[nm][key].values()],
                                   SEED0 + 61 + len(nm) + (0 if key == "ach" else 7), NB_BOOT)
        v = np.array(REPM[nm][key], float)
        sd = float(v.std(ddof=1)) if len(v) > 1 else float("nan")
        se = sd / np.sqrt(max(1, len(v)))
        NREP[nm][key] = dict(mean=float(v.mean()), lo=float(v.mean() - 1.96 * se),
                             hi=float(v.mean() + 1.96 * se), n=int(len(v)))

NULL_BLUR = []
def two_verdicts(nm, key, obs):
    """★ 보수적 문턱은 ✅ 를 어렵게 하는 건 맞지만 **❌ 를 만들어선 안 된다**.
    영점 자체의 CI 반폭이 관측 반폭보다 넓으면 그 비교는 **영점 잡음이 지배**한다
    → 기각을 **⚠️ 미결(영점 흐림)** 로 강등한다(R33 ① 을 영점에 적용)."""
    nhi = max(NSTAT[nm][key][2], NREP[nm][key]["hi"])
    thr = max(0.0, nhi) if np.isfinite(nhi) else float("nan")
    nh = max(mde(NSTAT[nm][key][1], NSTAT[nm][key][2]),
             mde(NREP[nm][key]["lo"], NREP[nm][key]["hi"]))
    blur = np.isfinite(nh) and np.isfinite(obs["mde"]) and nh > obs["mde"]
    def dem(v):
        if blur and v.startswith("❌"):
            NULL_BLUR.append((nm, key, float(nh), float(obs["mde"])))
            return "⚠️ 미결"
        return v
    return (dem(decide(obs["lo"], obs["hi"], thr, ">")), thr,
            dem(decide(obs["lo"], obs["hi"], nhi, ">")), nhi)

SRC = {"ach": ACH, "auc": AUC}
OBS, TAB = {}, {}
run.log(f"\n  {'대비':<13}{'Δ 달성률@300 ★1차':>28}{'배포':>7}{'Δ AUROC (2차)':>26}{'배포':>7}")
for nm, x_, y_ in CONTRASTS:
    OBS[nm], TAB[nm] = {}, {}
    row = f"  {nm:<13}"
    for key in ("ach", "auc"):
        m_, lo_, hi_, n_ = boot_pair([SRC[key][x_][r] for r in REC_OK],
                                     [SRC[key][y_][r] for r in REC_OK],
                                     SEED0 + 81 + len(nm) + (0 if key == "ach" else 7), NB_BOOT)
        o = dict(mean=m_, lo=lo_, hi=hi_, n=int(n_), mde=float(mde(lo_, hi_)))
        OBS[nm][key] = o
        vd, td, vm, tm = two_verdicts(nm, key, o)
        TAB[nm][key] = dict(obs=o, null_rec=list(NSTAT[nm][key][:3]), null_rep=NREP[nm][key],
                            thr_deploy=float(td), thr_mech=float(tm), v_deploy=vd, v_mech=vm)
        row += f"{m_:>+9.4f} [{lo_:+.4f},{hi_:+.4f}]{vd:>7}"
    run.log(row)

om = OBS[MAIN_CT][PRIMARY]; vd, td, vm, tm = two_verdicts(MAIN_CT, PRIMARY, om)
oa = OBS[MAIN_CT][SECONDARY]; vd2, td2, _, _ = two_verdicts(MAIN_CT, SECONDARY, oa)
run.log(f"\n  ★★★ **O2 주 관문** — `{MAIN_CT}` 차원 {FEAT['intv'].shape[1]} 동일")
run.log(f"    **1차 달성률@300** Δ **{om['mean']:+.4f}** [{om['lo']:+.4f}, {om['hi']:+.4f}] · "
        f"영점 {NSTAT[MAIN_CT][PRIMARY][0]:+.4f} · 문턱 {td:+.4f} · MDE {om['mde']:.4f} → {vd}")
run.log(f"    2차 매크로 AUROC Δ {oa['mean']:+.4f} [{oa['lo']:+.4f}, {oa['hi']:+.4f}] · "
        f"문턱 {td2:+.4f} → {vd2}")
run.log(f"    ▸ 두 지표가 갈리면 **Q4-J 와 같은 상황**이다 — 개입이 상위 꼬리에만 작용한 것")
if NULL_BLUR:
    run.log(f"    ⚠️ **영점 흐림으로 기각을 강등한 대비 {len(set(b[:2] for b in NULL_BLUR))}건** — "
            + " · ".join(f"{b[0]}/{b[1]} 영점반폭 {b[2]:.4f} > 관측반폭 {b[3]:.4f}"
                         for b in NULL_BLUR[:4])
            + " ⇒ 이건 **증거 없음**이지 **반대 증거가 아니다**(R33 ①)")
g_("O2", vd,
   f"`intv` vs `ishuf` 달성률 Δ {om['mean']:+.4f} [{om['lo']:+.4f}, {om['hi']:+.4f}] "
   f"(문턱 {td:+.4f}) {vd} · AUROC Δ {oa['mean']:+.4f} {vd2} — "
   + ("**P 의 시간(PR·PP·구간비)이 RR 너머 신호를 준다**" if vd.startswith("✅") else
      ("구간의 내용이 무작위 정렬과 다르지 않다" if vd.startswith("❌") else "가르지 못했다(R33 ①)")))

vf, tf, _, _ = two_verdicts("full-fshuf", PRIMARY, OBS["full-fshuf"][PRIMARY])
vmb, _, _, _ = two_verdicts("morph-base", PRIMARY, OBS["morph-base"][PRIMARY])
run.log(f"\n  ★★ **O3** — `full` vs `fshuf` 달성률 Δ "
        f"{OBS['full-fshuf'][PRIMARY]['mean']:+.4f} → {vf} · Q4-J 형태 재현 `morph − base` "
        f"{OBS['morph-base'][PRIMARY]['mean']:+.4f}(Q4-J +0.0944) → {vmb}")
g_("O3", vf,
   f"`full` vs `fshuf` 달성률 Δ {OBS['full-fshuf'][PRIMARY]['mean']:+.4f} · "
   f"`morph − base` {OBS['morph-base'][PRIMARY]['mean']:+.4f}(Q4-J 앵커 +0.0944 · 재현 "
   + ("일치" if abs(OBS['morph-base'][PRIMARY]['mean'] - 0.0944) < 0.03 else "불일치") + ")")
CONFIG["O2"] = TAB
run.save_json("config", CONFIG)


In [ ]:
# CELL 5 — 【O-D】 ★★ O4 사용자 가설 (a)~(d) 개별 판정
run.log("\n" + "=" * 100)
run.log("【O-D】 ★★ O4 — 임상 가설을 **블록별로** 판정한다")
run.log("=" * 100)
BLOCKS = {
    "(a) P-P/ΔPR":    [1, 2, 3],            # dpr_ms · pr_dev_ms · pp_over_rr
    "(b) P폭/QRS폭":   [4],
    "(c) P폭/T폭":     [5],
    "(d) TP(정규화)":  [6, 7],
    "(e) P 형태·검출": [0, 8, 9, 10, 11],
}
run.log(f"  각 블록만 base 에 더한 팔을 LORO 로 돌린다(선택 편의 없음 · R36 ②)")
run.log(f"\n  {'블록':<16}{'차원':>5}{'Δ 달성률':>22}{'Δ AUROC':>22}{'최고 R²':>9}")
O4 = {}
for nm, cols in BLOCKS.items():
    Xb = np.c_[F_BASE, INTV[:, cols]]
    Lb, _ = loro(Xb, None, tag=f"blk_{nm}")
    ab = {r: at_k(Lb, r, MAIN_K)["ach"] for r in REC_OK}
    ub = per_auc(Lb)
    da = boot_pair([ACH["base"][r] for r in REC_OK], [ab[r] for r in REC_OK],
                   SEED0 + 200 + len(nm), NB_BOOT)
    du = boot_pair([AUC["base"][r] for r in REC_OK], [ub[r] for r in REC_OK],
                   SEED0 + 230 + len(nm), NB_BOOT)
    r2max = max(DUP[INAMES[j]]["r2"] for j in cols)
    O4[nm] = dict(cols=[INAMES[j] for j in cols], ach=list(da[:3]), auc=list(du[:3]),
                  r2max=float(r2max))
    run.log(f"  {nm:<16}{len(cols):>5}{da[0]:>+8.4f} [{da[1]:+.4f},{da[2]:+.4f}]"
            f"{du[0]:>+8.4f} [{du[1]:+.4f},{du[2]:+.4f}]{r2max:>9.3f}")
best_b = max(BLOCKS, key=lambda n: O4[n]["ach"][0])
run.log(f"\n  ▸ 달성률 기준 최선 블록 **{best_b}** ({O4[best_b]['ach'][0]:+.4f}) — "
        f"열 {O4[best_b]['cols']}")
run.log(f"  ▸ ⚠️ 블록별 비교는 **탐색**이다(관문 아님) — 주 관문은 12열 통째 넣은 O2 다(R36 ②)")
g_("O4", "(관문 아님)",
   " · ".join(f"{n} 달성률 {O4[n]['ach'][0]:+.4f}" for n in BLOCKS) + f" · 최선 {best_b}")
CONFIG["O4"] = O4
run.save_json("config", CONFIG)


In [ ]:
# CELL 6 — 【O-E】 O5 딥러닝 5-겹(GPU 있을 때만) · 오류 해부 재측
run.log("\n" + "=" * 100)
run.log("【O-E】 O5 — 딥러닝 절(**관문 아님** · GPU 없으면 건너뛴다)")
run.log("=" * 100)
DLR = {"ran": False, "reason": ""}
try:
    import torch
    import torch.nn as nn
    HAS_CUDA = torch.cuda.is_available()
except Exception as e:
    torch = None; HAS_CUDA = False; DLR["reason"] = f"torch 없음({e})"
if torch is None:
    run.log(f"  ⏭️ 건너뜀 — {DLR['reason']}")
elif not HAS_CUDA and not SMOKE:
    DLR["reason"] = "CUDA 없음 — CPU 로는 5겹 × 4에폭이 너무 느리다"
    run.log(f"  ⏭️ 건너뜀 — {DLR['reason']}. **런타임을 GPU 로 바꾸면 이 절이 돈다**")
else:
    dev = "cuda" if HAS_CUDA else "cpu"
    run.log(f"  ▶ 장치 **{dev}** · {DL_FOLDS}겹 · {DL_EPOCH}에폭 · 임베딩 {DL_EMB}")
    ordr = sorted(REC_OK, key=lambda r: (BURD[r], r))
    FOLD = {r: i % DL_FOLDS for i, r in enumerate(ordr)}
    class Net(nn.Module):
        def __init__(self, n_extra):
            super().__init__()
            self.c = nn.Sequential(
                nn.Conv1d(NL, 16, 7, 2, 3), nn.BatchNorm1d(16), nn.ReLU(),
                nn.Conv1d(16, 32, 5, 2, 2), nn.BatchNorm1d(32), nn.ReLU(),
                nn.Conv1d(32, 32, 3, 2, 1), nn.BatchNorm1d(32), nn.ReLU(),
                nn.AdaptiveAvgPool1d(1))
            self.e = nn.Linear(32, DL_EMB)
            self.h = nn.Linear(DL_EMB + n_extra, 1)
        def forward(self, w, x):
            z = torch.relu(self.e(self.c(w).squeeze(-1)))
            return self.h(torch.cat([z, x], 1) if x is not None and x.shape[1] else z).squeeze(-1)
    def dl_fold(use_extra):
        sc = np.full(len(K), np.nan)
        Xe = FEAT["full"] if use_extra else np.zeros((len(K), 0))
        for f in range(DL_FOLDS):
            te_r = [r for r in REC_OK if FOLD[r] == f]
            tr_r = [r for r in REC_OK if FOLD[r] != f]
            tr = np.concatenate([IDXS[r] for r in tr_r]); te = np.concatenate([IDXS[r] for r in te_r])
            mu, sd = Xe[tr].mean(0) if Xe.shape[1] else 0.0, (Xe[tr].std(0) + 1e-9) if Xe.shape[1] else 1.0
            wmu = BEAT[tr].mean(); wsd = BEAT[tr].std() + 1e-9
            net = Net(Xe.shape[1]).to(dev)
            opt = torch.optim.Adam(net.parameters(), 1e-3)
            pw = torch.tensor([(~TT_[tr]).sum() / max(1, TT_[tr].sum())], dtype=torch.float32,
                              device=dev)
            lossf = nn.BCEWithLogitsLoss(pos_weight=pw)
            for ep in range(DL_EPOCH):
                net.train(); perm = np.random.RandomState(SEED0 + f * 10 + ep).permutation(len(tr))
                for b0 in range(0, len(tr), DL_BATCH):
                    bi = tr[perm[b0:b0 + DL_BATCH]]
                    w = torch.tensor((BEAT[bi] - wmu) / wsd, dtype=torch.float32, device=dev)
                    x = torch.tensor((Xe[bi] - mu) / sd, dtype=torch.float32, device=dev) \
                        if Xe.shape[1] else torch.zeros((len(bi), 0), device=dev)
                    y = torch.tensor(TT_[bi].astype("float32"), device=dev)
                    opt.zero_grad(); l = lossf(net(w, x), y); l.backward(); opt.step()
            net.eval()
            with torch.no_grad():
                for b0 in range(0, len(te), 4096):
                    bi = te[b0:b0 + 4096]
                    w = torch.tensor((BEAT[bi] - wmu) / wsd, dtype=torch.float32, device=dev)
                    x = torch.tensor((Xe[bi] - mu) / sd, dtype=torch.float32, device=dev) \
                        if Xe.shape[1] else torch.zeros((len(bi), 0), device=dev)
                    sc[bi] = net(w, x).cpu().numpy()
            run.log(f"    ({time.time()-T0:>5.0f}초) {'hybrid' if use_extra else 'wave'} "
                    f"겹 {f+1}/{DL_FOLDS}")
        return sc
    def cpu_fold(X):
        sc = np.full(len(K), np.nan)
        for f in range(DL_FOLDS):
            te_r = [r for r in REC_OK if FOLD[r] == f]
            tr_r = [r for r in REC_OK if FOLD[r] != f]
            tr = np.concatenate([IDXS[r] for r in tr_r]); te = np.concatenate([IDXS[r] for r in te_r])
            mu, sd = X[tr].mean(0), X[tr].std(0) + 1e-9
            lr = LogisticRegression(max_iter=3000, C=1.0).fit((X[tr] - mu) / sd, TT_[tr].astype(int))
            sc[te] = lr.decision_function((X[te] - mu) / sd)
        return sc
    DLS = {"cpu_base": cpu_fold(FEAT["base"]), "cpu_full": cpu_fold(FEAT["full"]),
           "dl_wave": dl_fold(False), "dl_hybrid": dl_fold(True)}
    run.log(f"\n  {'팔(5겹 · 같은 자)':<18}{'달성률@300':>13}{'매크로 AUROC':>15}{'PR-AUC':>10}")
    DLR["res"] = {}
    for nm, sc in DLS.items():
        a_ = float(np.mean([at_k(sc, r, MAIN_K)["ach"] for r in REC_OK]))
        u_ = float(np.mean(list(per_auc(sc).values())))
        p_ = float(np.mean([average_precision_score(TT_[IDXS[r]].astype(int), sc[IDXS[r]])
                            for r in REC_OK]))
        DLR["res"][nm] = dict(ach=a_, auc=u_, ap=p_)
        run.log(f"  {nm:<18}{a_:>13.4f}{u_:>15.4f}{p_:>10.4f}")
    dh = boot_pair([at_k(DLS["cpu_full"], r, MAIN_K)["ach"] for r in REC_OK],
                   [at_k(DLS["dl_hybrid"], r, MAIN_K)["ach"] for r in REC_OK],
                   SEED0 + 301, NB_BOOT)
    run.log(f"  ▸ `dl_hybrid − cpu_full` 달성률 Δ **{dh[0]:+.4f}** [{dh[1]:+.4f}, {dh[2]:+.4f}]")
    run.log(f"  ▸ 문헌 — 환자분리 SVEB 는 1D CNN 도 **74.56%** 로 20년째 ~75% 다")
    DLR["ran"] = True; DLR["gain"] = list(dh[:3])
g_("O5", "(관문 아님)",
   (f"5겹 — " + " · ".join(f"{k} 달성률 {v['ach']:.4f}" for k, v in DLR["res"].items())
    + f" · hybrid−full {DLR['gain'][0]:+.4f}") if DLR["ran"] else f"건너뜀 ({DLR['reason']})")

# ── 오류 해부 재측(형태·구간이 위양성 구성을 어떻게 바꿨나)
VSET, NSET = ("V", "E", "F"), ("N", "L", "R", "e", "j", "n")
FPC = {}
for a in ("base", "morph", "intv", "full"):
    c = Counter()
    for r in REC_OK:
        idx = IDXS[r]; bad = R300[a][r]["flag"] & (~TT_[idx])
        for s_ in SYM[idx][bad]: c[str(s_)] += 1
    FPC[a] = dict(v=sum(c[s] for s in VSET), n=sum(c[s] for s in NSET), tot=sum(c.values()))
run.log(f"\n  ▸ **위양성 구성 재측**  {'팔':<8}{'심실기원':>10}{'상심실정상':>12}{'전체':>9}")
for a, d in FPC.items():
    run.log(f"                        {a:<8}{d['v']:>10}{d['n']:>12}{d['tot']:>9}")
run.log(f"    (Q4-J 앵커 — base V {REF['q4j']['fp_v']} · N {REF['q4j']['fp_n']} → morph V "
        f"{REF['q4j']['fp_v_after']} · N {REF['q4j']['fp_n_after']})")
CONFIG["O5"] = DLR; CONFIG["fp"] = FPC
run.save_json("config", CONFIG)


In [ ]:
# CELL 7 — 【O-F】 필요표본 · O6 검산표 · 그림 · 요약
run.log("\n" + "=" * 100)
run.log("【O-F】 필요표본 · O6 검산표")
run.log("=" * 100)
eff = om["mean"] - TAB[MAIN_CT][PRIMARY]["thr_deploy"]
n5 = need_super(NRE, om["mde"], eff); n8 = need_super(NRE, om["mde"], eff, True)
bad = (not np.isfinite(eff)) or abs(eff) < om["mde"]
run.log(f"  O2 주 관문(달성률) 효과-문턱 {eff:+.4f} · 반폭 {om['mde']:.4f} · n(50%) {n5:.0f} · "
        f"n(80%) {n8:.0f}  "
        + ("★ **해석 불가**(R41 ②)" if bad else
           ("이미 충분하다" if n8 <= NRE else "표본이 더 필요하다")))

CHECK = [
    dict(claim="★★★ Q4-J 는 성공했는데 **내가 잘못된 자로 쟀다**",
         num=f"Q4-J 형태 — 달성률 +0.0944(+11.7%) · PR-AUC +0.1440(+24.8%) · PPV +0.0660 인데 "
             f"내가 건 매크로 AUROC 는 +0.0109 뿐이라 ⚠️ 미결이 났다. 본 런 재현 "
             f"`morph − base` 달성률 {OBS['morph-base']['ach']['mean']:+.4f}",
         assume="**없음** — Q4-J 실측이다",
         iffalse="★★★ 개입이 **상위 꼬리**에 작용하면 전체 순위 지표는 못 본다. Q4-G 의 "
                 "ρ(AUROC, 달성률)=+0.8869 는 **레코드 간 상관**이지 개입 효과가 아니다"),
    dict(claim=f"★★★ O2 주 관문 — 구간 12열의 달성률 값 {om['mean']:+.4f} → {VERD['O2']}",
         num=f"두 팔 모두 {FEAT['intv'].shape[1]}차원 · 구간 블록만 레코드 안에서 공동 행치환 · "
             f"영점 {NSTAT[MAIN_CT][PRIMARY][0]:+.4f}(rep {NREP[MAIN_CT][PRIMARY]['mean']:+.4f}) · "
             f"AUROC 로는 {oa['mean']:+.4f}",
         assume="델리네이션·템플릿·잔차화 전부 **라벨을 안 쓴다**(R22)",
         iffalse="★★ 두 지표가 갈리면 그 자체가 **개입이 상위 꼬리에만 작용했다**는 증거다"),
    dict(claim="★★★ **PP(i) = RR(i) − ΔPR(i)** — P-P 를 따로 재는 건 ΔPR 을 재는 것과 같다",
         num=f"대수적 항등(검증 max|Δ| = 0.0). 블록 (a) 달성률 {O4['(a) P-P/ΔPR']['ach'][0]:+.4f} · "
             f"최고 R²(base) {O4['(a) P-P/ΔPR']['r2max']:.3f}",
         assume="P 위치는 **템플릿 P 조각과의 교차상관 최대**로 잡는다(±83ms)",
         iffalse="★★ 이소성 초점은 SA node 와 다른 자리라 심방내 전도 경로가 달라 PR 이 바뀐다 — "
                 "**RR 과 직교**하는 축이라는 게 이 특징의 근거다"),
    dict(claim="★★★ TP 구간은 **그대로 넣으면 RR 의 재표현**이다 — 잔차화로만 넣었다",
         num=f"`RR = TP+P+PR+QRS+ST+T` · 합성 확인 ρ(TP,RR)=+1.0000 · 잔차 SD 3.1e-14. "
             f"본 런 `tp_over_rr` R² {DUP['tp_over_rr']['r2']:.3f} · `tp_resid` R² "
             f"{DUP['tp_resid']['r2']:.3f} · 블록 (d) 달성률 {O4['(d) TP(정규화)']['ach'][0]:+.4f}",
         assume="잔차 회귀는 **레코드 안에서 라벨 없이** 적합한다",
         iffalse="★★★ 이게 Q4-H 가 죽은 그 자리다 — **중복 감사(O1)가 그걸 입구에서 잡는 자**다"),
    dict(claim=f"★★ **중복 감사** — 중복 {sum(1 for v in DUP.values() if v['verdict'].startswith('⛔'))} · "
               f"레코드상수 {sum(1 for v in DUP.values() if v['verdict'].startswith('⚠️'))}",
         num=" · ".join(f"{n} R²{DUP[n]['r2']:.2f}/분산{DUP[n]['within']:.2f}"
                        for n in INAMES[:6]),
         assume=f"문턱 R² > {DUP_R2} 를 **사전 고정**(R34 ②) · 영점 흐림 강등 "
                f"{len(set(b[:2] for b in NULL_BLUR))}건",
         iffalse="★ 레코드 내 분산비가 낮은 열은 **레코드 상수**라 within-record 순위를 못 바꾼다 — "
                 "층② 불가능 정리와 같은 이유다"),
    dict(claim="★ QRS 폭 정의를 **표준으로 고쳤다**",
         num=f"원안은 「Q 시작 − S **시작**」이었으나 표준 QRS 폭은 **Q 시작 ~ S 종료(J point)** 다. "
             f"창 {W_Q_S} · 임계 {FRAC_QRS} · 템플릿 QRS 폭 중앙 "
             f"{np.median([(DELIN[int(r)]['q'][1]-DELIN[int(r)]['q'][0])/FS*1000 for r in RS]):.1f}ms",
         assume="템플릿에서 델리네이션하고 박동은 같은 창에서 잰다",
         iffalse="★ 폭 중앙이 생리 범위(대략 60~120ms)를 크게 벗어나면 델리네이션이 틀린 것이다"),
    dict(claim=("★ 딥러닝 5겹 — " + (f"hybrid−full 달성률 {DLR['gain'][0]:+.4f}"
                                    if DLR["ran"] else f"건너뜀({DLR['reason']})")),
         num=(" · ".join(f"{k} {v['ach']:.4f}" for k, v in DLR["res"].items())
              if DLR["ran"] else "GPU 런타임에서 다시 돌리면 된다"),
         assume="**같은 5겹에서 CPU 팔을 다시 재서** 비교한다 — LORO 수치와 직접 비교하지 않는다",
         iffalse="★ 문헌상 환자분리 SVEB 는 1D CNN 도 74.56% 라 **큰 기대를 안 하는 게** 사전 입장이다"),
]
for i, ck in enumerate(CHECK, 1):
    run.log(f"\n  [{i}] **{ck['claim']}**")
    run.log(f"      근거   {ck['num']}")
    run.log(f"      가정   {ck['assume']}")
    run.log(f"      틀리면 {ck['iffalse']}")
CONFIG["need"] = dict(effect=float(eff), half=float(om["mde"]), sup50=float(n5),
                      sup80=float(n8), uninterpretable=bool(bad))
CONFIG["O6"] = CHECK
run.save_json("config", CONFIG)

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image, display
fig, ax = plt.subplots(1, 3, figsize=(16.5, 4.6))
xs = np.arange(len(ARMS))
ax[0].bar(xs - 0.2, [np.mean(list(ACH[a].values())) for a in ARMS], 0.4, label="achievement@300")
ax[0].bar(xs + 0.2, [np.mean(list(AUC[a].values())) for a in ARMS], 0.4, label="macro AUROC")
ax[0].set_xticks(xs); ax[0].set_xticklabels(ARMS, fontsize=8, rotation=20)
ax[0].set_ylim(0.6, 1.0); ax[0].legend(fontsize=7); ax[0].grid(alpha=.3, axis="y")
ax[0].set_title("primary (achievement) vs secondary (AUROC)", fontsize=9)

nm = [c[0] for c in CONTRASTS]
vv = [OBS[n][PRIMARY]["mean"] for n in nm]
lo = [vv[i] - OBS[n][PRIMARY]["lo"] for i, n in enumerate(nm)]
hi = [OBS[n][PRIMARY]["hi"] - vv[i] for i, n in enumerate(nm)]
ax[1].errorbar(vv, np.arange(len(nm)), xerr=[lo, hi], fmt="o", capsize=5, color="tab:blue")
ax[1].scatter([NSTAT[n][PRIMARY][0] for n in nm], np.arange(len(nm)), marker="x", s=45,
              color="tab:gray", label="null (raw)")
ax[1].axvline(0, color="k", lw=.9)
ax[1].set_yticks(range(len(nm))); ax[1].set_yticklabels(nm, fontsize=8)
ax[1].set_xlabel("achievement@300 delta"); ax[1].legend(fontsize=7)
ax[1].set_title("O2/O3 : contrasts on the primary metric", fontsize=9)
ax[1].grid(alpha=.3, axis="x")

bn = list(BLOCKS.keys())
ax[2].barh(np.arange(len(bn)), [O4[b]["ach"][0] for b in bn], color="tab:green")
ax[2].set_yticks(range(len(bn)))
ax[2].set_yticklabels(["(a) dPR/PP", "(b) Pw/QRSw", "(c) Pw/Tw", "(d) TP norm",
                       "(e) P morph"][:len(bn)], fontsize=8)
ax[2].axvline(0, color="k", lw=.9); ax[2].set_xlabel("achievement@300 delta vs base")
ax[2].set_title("O4 : which clinical hypothesis pays", fontsize=9); ax[2].grid(alpha=.3, axis="x")
fig.tight_layout()
PNG = run.save_fig("q4k_pp_interval", fig)
plt.close(fig); display(Image(PNG))

run.log("\n" + "=" * 100)
run.log("요약")
run.log("=" * 100)
ok_ = lambda k: VERD.get(k, "").startswith("✅")
for g in READ_ORDER[:6]:
    run.log(f"  {g:<5}{VERD.get(g, '(관문 아님)')}")
run.log("")
run.log(f"  ★★★ **지표 정정** — 주 지표를 **달성률@300** 으로 바꿨다(Q4-J 에서 형태가 달성률 "
        f"+0.0944 인데 AUROC 는 +0.0109 였다)")
run.log(f"  ★★★ **O2 주 관문** — `intv` vs `ishuf` 달성률 {om['mean']:+.4f} "
        f"[{om['lo']:+.4f}, {om['hi']:+.4f}] → {VERD['O2']} (AUROC {oa['mean']:+.4f})")
run.log(f"  ★★ **O3** — `full` vs `fshuf` {OBS['full-fshuf'][PRIMARY]['mean']:+.4f} → {VERD['O3']}")
run.log(f"  ★★ **O4 임상 가설** — " + " · ".join(f"{n} {O4[n]['ach'][0]:+.4f}" for n in BLOCKS))
run.log(f"  ★★ **O1 중복 감사** — 중복 "
        f"{sum(1 for v in DUP.values() if v['verdict'].startswith('⛔'))} · 레코드상수 "
        f"{sum(1 for v in DUP.values() if v['verdict'].startswith('⚠️'))} / {len(DUP)}열")
run.log(f"  ▸ 달성률@300 — " + " · ".join(f"{a} {np.mean(list(ACH[a].values())):.4f}" for a in ARMS))
run.log(f"  ▸ 위양성 심실기원 — " + " · ".join(f"{a} {FPC[a]['v']}" for a in FPC))
run.log(f"  ▸ DL — " + ("돌았다" if DLR["ran"] else f"건너뜀({DLR['reason']})"))

run.finish({
    "exp_id": "quest46_q4k_pp_interval",
    "metric": "achievement300_intv_minus_ishuf",
    "value": float(om["mean"]),
    "passed": bool(ok_("O0") and ok_("O2")),
    "summary": ("Q4-J 는 성공했는데 내가 잘못된 자로 쟀다 — 형태 8열이 달성률을 +0.0944"
                "(+11.7%) · PR-AUC +0.1440 올렸는데 내가 주 관문으로 건 매크로 AUROC 는 "
                "+0.0109 뿐이라 미결이 났다. 개입이 **상위 300개에서 V 를 걷어내는**(V 위양성 "
                "4380 → 1444) 일이라 전체 순위 지표가 못 본 것이다. 그래서 **주 지표를 "
                "달성률@300 으로 바꾸고**(데이터 보기 전 사전등록) 사용자 임상 가설을 "
                "수치화했다: (a) **P-P interval** — 대수적으로 `PP = RR − ΔPR` 이므로 「P-P 가 "
                "R-R 과 어긋나는 지점」은 정확히 ΔPR 이고, 이소성 초점은 심방내 전도 경로가 "
                "달라 PR 이 바뀐다(RR 과 직교) (b) **P폭/QRS폭** — 전도 속도 비. QRS 정의를 "
                "표준(Q 시작~S 종료)으로 고쳤다 (c) **P폭/T폭** (d) **TP 구간** — 그대로 넣으면 "
                "`RR = TP+P+PR+QRS+ST+T` 라 RR 의 재표현이 되므로(합성 ρ=+1.0000) **RR 잔차화**"
                "로만 넣었다. 그리고 **중복 감사**라는 새 자를 지었다 — 모든 새 열을 base 9열에 "
                "회귀시켜 R² 를 먼저 낸다. Q4-H 를 사후에야 안 실수를 입구에서 잡는다. "
                "딥러닝 절은 GPU 가 있을 때만 같은 5겹에서 CPU 팔과 함께 돈다."),
    "verdicts": VERD, "notes": NOTE, "rule_check": RULE_CHECK,
    "cohort": CONFIG.get("cohort", {}), "O0": CONFIG.get("O0", {}),
    "O1": CONFIG.get("O1", {}), "O2": CONFIG.get("O2", {}), "O4": CONFIG.get("O4", {}),
    "O5": CONFIG.get("O5", {}), "fp": CONFIG.get("fp", {}), "need": CONFIG.get("need", {}),
    "O6": CONFIG.get("O6", []), "fig": PNG})
run.log(f"\n저장 완료 — {run.dir}")
run.log("다음: `python pipelines/ingest_run.py --results result.json "
        "--notebook notebooks/quest46_q4k_pp_interval.ipynb`")
